# TGDSA: Two-Grid Diffusion Synthetic Acceleration

![Graphite block with a uniform fast source and vacuum boundaries](images/tgdsa_graphite_block.png)

This tutorial adds two-grid diffusion synthetic acceleration (TGDSA) to a multigroup graphite calculation. Two otherwise identical solves show that TGDSA changes the convergence process without changing the transport solution.

## Graphite problem

A uniform fast source is applied to a 40 cm by 40 cm graphite block with vacuum boundaries. The 168-group graphite cross sections come from `test/assets/xs/xs_graphite_pure.xs`, which is also used by the OpenSn regression suite. The file contains higher-order scattering data; this compact example retains the first two Legendre moments ($P_1$).

All 168 groups are placed in one groupset. The baseline uses ordinary Richardson source iteration with no acceleration. The second run changes only `apply_tgdsa`, so it uses TGDSA without WGDSA or another transport accelerator. TGDSA collapses the multigroup error to a one-group diffusion problem and projects the correction back to every group.

In [ ]:
from pathlib import Path

from mpi4py import MPI
from pyopensn.aquad import GLCProductQuadrature2DXY
from pyopensn.context import Finalize
from pyopensn.mesh import KBAGraphPartitioner, OrthogonalMeshGenerator
from pyopensn.post import VolumePostprocessor
from pyopensn.solver import DiscreteOrdinatesProblem, SteadyStateSourceSolver
from pyopensn.source import VolumetricSource
from pyopensn.xs import MultiGroupXS

comm = MPI.COMM_WORLD
rank = comm.rank
tutorial_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
repo_root = next(
    path for path in (tutorial_dir, *tutorial_dir.parents)
    if (path / "test/assets/xs").is_dir()
)

## Build and solve both cases

The 10 by 10 mesh uses a 2 by 2 KBA partition, so run the generated script with four MPI processes. Both cases use the same classic Richardson source iteration and convergence tolerance. The Boolean argument controls only `apply_tgdsa`.

In [ ]:
def solve(use_tgdsa):
    nodes = [4.0 * i for i in range(11)]
    partitioner = KBAGraphPartitioner(
        nx=2, ny=2, xcuts=[20.0], ycuts=[20.0]
    )
    mesh = OrthogonalMeshGenerator(
        node_sets=[nodes, nodes], partitioner=partitioner
    ).Execute()
    mesh.SetOrthogonalBoundaries()
    mesh.SetUniformBlockID(0)

    graphite_xs = MultiGroupXS()
    graphite_xs.LoadFromOpenSn(
        str(repo_root / "test/assets/xs/xs_graphite_pure.xs")
    )
    num_groups = graphite_xs.num_groups
    source_strength = [0.0] * num_groups
    source_strength[0] = 1.0

    quadrature = GLCProductQuadrature2DXY(
        n_polar=2, n_azimuthal=4, scattering_order=1
    )
    groupset = {
        "groups_from_to": (0, num_groups - 1),
        "angular_quadrature": quadrature,
        "inner_linear_method": "classic_richardson",
        "l_abs_tol": 1.0e-8,
        "l_max_its": 1200,
    }
    if use_tgdsa:
        groupset.update(
            {
                "apply_tgdsa": True,
                "tgdsa_l_abs_tol": 1.0e-8,
                "tgdsa_l_max_its": 200,
                "tgdsa_solver_policy": "auto",
            }
        )

    problem = DiscreteOrdinatesProblem(
        mesh=mesh,
        num_groups=num_groups,
        groupsets=[groupset],
        xs_map=[{"block_ids": [0], "xs": graphite_xs}],
        volumetric_sources=[
            VolumetricSource(block_ids=[0], group_strength=source_strength)
        ],
        boundary_conditions=[
            {"name": name, "type": "vacuum"}
            for name in ("xmin", "xmax", "ymin", "ymax")
        ],
        options={"verbose_inner_iterations": False},
    )

    solver = SteadyStateSourceSolver(problem=problem)
    comm.Barrier()
    start = MPI.Wtime()
    solver.Initialize()
    solver.Execute()
    comm.Barrier()
    elapsed = comm.allreduce(MPI.Wtime() - start, op=MPI.MAX)

    average = VolumePostprocessor(problem=problem, value_type="avg")
    average.Execute()
    group_flux = average.GetValue()[0]
    return sum(group_flux), group_flux[120], solver.GetNumSweeps(), elapsed


unaccelerated_total, unaccelerated_g120, unaccelerated_sweeps, unaccelerated_time = (
    solve(False)
)
tgdsa_total, tgdsa_g120, tgdsa_sweeps, tgdsa_time = solve(True)

## Compare the solutions

The primary correctness metric is the relative difference in the energy-summed, volume-averaged scalar flux. Group 120 provides a second multigroup check. Transport sweeps verify that TGDSA reduces high-order work. We also report the maximum wall time across ranks and its speedup; timing is illustrative rather than a regression metric because it depends on the machine.

In [ ]:
total_relative_difference = (
    abs(tgdsa_total - unaccelerated_total) / abs(unaccelerated_total)
)
g120_relative_difference = (
    abs(tgdsa_g120 - unaccelerated_g120) / abs(unaccelerated_g120)
)
speedup = unaccelerated_time / tgdsa_time

if rank == 0:
    print(f"Unaccelerated total average flux={unaccelerated_total:.12e}")
    print(f"TGDSA total average flux={tgdsa_total:.12e}")
    print(f"TGDSA total flux relative difference={total_relative_difference:.12e}")
    print(f"TGDSA group 120 relative difference={g120_relative_difference:.12e}")
    print(f"Unaccelerated transport sweeps={unaccelerated_sweeps}")
    print(f"TGDSA transport sweeps={tgdsa_sweeps}")
    print(f"Unaccelerated wall time (s)={unaccelerated_time:.6f}")
    print(f"TGDSA wall time (s)={tgdsa_time:.6f}")
    print(f"TGDSA speedup={speedup:.6f}")

assert total_relative_difference < 1.0e-6
assert g120_relative_difference < 1.0e-6
assert tgdsa_sweeps < unaccelerated_sweeps

A representative four-process run gives the following solution comparison:

| Solve | Energy-summed average flux | Group-120 average flux |
|---|---:|---:|
| No acceleration | 42.31957467 | $1.7956985\times10^{-2}$ |
| TGDSA | 42.31957468 | $1.7956985\times10^{-2}$ |

The same run gives the convergence and timing comparison:

| Solve | Transport sweeps | Wall time (s) |
|---|---:|---:|
| No acceleration | 1010 | 8.32 |
| TGDSA | 459 | 4.38 |

The sweep counts come directly from `solver.GetNumSweeps()`. TGDSA reduces transport sweeps by approximately 55% and gives a representative speedup of 1.90. Exact sweep counts can differ slightly with floating-point ordering, and wall time is machine-dependent. The converged total fluxes agree to about $4\times10^{-10}$ relative error.

## Finalize (for Jupyter Notebook only)

In script mode, PyOpenSn handles finalization automatically. In a Jupyter kernel, finalize OpenSn before MPI.

In [ ]:
if "opensn_console" not in globals():
    from IPython import get_ipython

    if get_ipython() is not None:
        Finalize()
        MPI.Finalize()